<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GLM_GPT_AOCC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# 🚀 AOC HYBRID SYSTEM: GPT-5.6 + GLM-5.2 INTEGRATION
# ============================================================================
# This notebook combines:
#   1. GPT-5.6: SOL (deep), TERRA (balanced), LUNA (fast)
#   2. GLM-5.2: Thinking/Reasoning with 6-step CoT
#   3. Intelligent routing for optimal performance
# ============================================================================

# ============================================================================
# CELL 1: SETUP & API KEYS
# ============================================================================

from google.colab import userdata
import os

# Set API keys from Colab secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['ZAI_KEY'] = userdata.get('ZAI_KEY')

print("=" * 80)
print("🚀 AOC HYBRID SYSTEM INITIALIZING")
print("=" * 80)
print(f"✅ OpenAI API Key: {'*' * 10}{os.environ['OPENAI_API_KEY'][-4:] if os.environ.get('OPENAI_API_KEY') else 'NOT FOUND'}")
print(f"✅ Z.ai API Key: {'*' * 10}{os.environ['ZAI_KEY'][-4:] if os.environ.get('ZAI_KEY') else 'NOT FOUND'}")
print("=" * 80)

# ============================================================================
# CELL 2: IMPORTS
# ============================================================================

import json
import time
import logging
import random
import re
import asyncio
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass, asdict, field
from enum import Enum
from functools import lru_cache

from openai import OpenAI, AsyncOpenAI

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports loaded")

# ============================================================================
# CELL 3: DATA CLASSES
# ============================================================================

class Priority(Enum):
    CRITICAL = 1
    HIGH = 2
    MEDIUM = 3
    LOW = 4

class IncidentType(Enum):
    FLIGHT_DELAY = "flight_delay"
    CREW_UNAVAILABLE = "crew_unavailable"
    WEATHER = "weather"
    MAINTENANCE = "maintenance"
    PASSENGER_ISSUE = "passenger_issue"
    REGULATORY = "regulatory"
    ATC = "atc"
    FUEL = "fuel"
    SECURITY = "security"
    MEDICAL = "medical"

@dataclass
class Flight:
    flight_number: str
    departure: str
    arrival: str
    departure_time: str
    arrival_time: str
    status: str
    gate: str
    aircraft: str
    passengers: int
    crew: List[str]

@dataclass
class Incident:
    id: str
    type: IncidentType
    severity: str  # CRITICAL, HIGH, MEDIUM, LOW
    flight: Flight
    description: str
    impact: str
    timestamp: str
    status: str  # OPEN, IN_PROGRESS, RESOLVED
    owner: str
    actions_taken: List[str] = field(default_factory=list)
    extra_data: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ThinkingResult:
    """GLM-5.2 thinking mode result"""
    thinking: str
    thinking_words: int
    has_structured: bool
    total_time: float
    priority: str
    timestamp: str
    model_used: str
    recommendation: str
    risk_assessment: Dict
    estimated_cost: str
    justification: str
    timeline: Dict

@dataclass
class Decision:
    incident_id: str
    recommendation: str
    actions: List[str]
    priority: Priority
    estimated_cost: float
    timeline: str
    risk_level: str
    confidence: float
    reasoning: str
    model_used: str
    timestamp: str
    execution_time: float = 0.0
    thinking_result: Optional[ThinkingResult] = None

@dataclass
class AgentState:
    incidents: List[Incident] = field(default_factory=list)
    decisions: List[Decision] = field(default_factory=list)
    active_flights: List[Flight] = field(default_factory=list)
    last_update: str = datetime.now().isoformat()
    metrics: Dict[str, Any] = field(default_factory=dict)

# ============================================================================
# CELL 4: GLM-5.2 THINKING ENGINE
# ============================================================================

class GLMThinkingEngine:
    """GLM-5.2 Thinking Mode Engine with 6-step Chain-of-Thought"""

    def __init__(self, api_key: str):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.z.ai/api/paas/v4/"
        )
        self.model = "glm-5.2"
        self.section_headers = [
            "STEP 1: SITUATION ANALYSIS",
            "STEP 2: DATA ASSESSMENT",
            "STEP 3: RISK EVALUATION",
            "STEP 4: OPTIONS ANALYSIS",
            "STEP 5: TRADE-OFF ASSESSMENT",
            "STEP 6: DECISION RATIONALE"
        ]
        print("🧠 GLM-5.2 Thinking Engine initialized")

    def build_context(self, incident: Incident) -> str:
        """Build detailed context for GLM thinking"""
        flight = incident.flight
        extra = incident.extra_data

        return f"""
FLIGHT OPERATIONS DATA
======================

FLIGHT INFORMATION:
- Flight Number: {flight.flight_number}
- Status: {flight.status}
- Departure: {flight.departure} → {flight.arrival}
- Scheduled Departure: {flight.departure_time}
- Gate: {flight.gate}
- Aircraft: {flight.aircraft}
- Passengers: {flight.passengers}

INCIDENT DETAILS:
- Type: {incident.type.value}
- Severity: {incident.severity}
- Description: {incident.description}
- Impact: {incident.impact}

ADDITIONAL DATA:
{self._format_extra_data(extra)}
"""

    def _format_extra_data(self, data: Dict) -> str:
        if not data:
            return "- No additional data"
        return '\n'.join(f"- {k}: {v}" for k, v in data.items())

    def think(self, incident: Incident, priority: str = "balanced") -> ThinkingResult:
        """Run GLM-5.2 thinking mode"""

        context = self.build_context(incident)

        prompt = f"""
**CRITICAL INSTRUCTION: You MUST show your complete step-by-step reasoning process BEFORE giving any final recommendation.**

You are a senior aviation operations manager with 20+ years of experience. Analyze this situation thoroughly and systematically.

{context}

**PRIORITY:** {priority.upper()}

**REQUIRED OUTPUT FORMAT:**

You MUST follow this exact structure:

---
STEP 1: SITUATION ANALYSIS
- What is the current flight status?
- What are the key operational parameters?
- What is the context of this situation?
- What are the immediate concerns?

STEP 2: DATA ASSESSMENT
- Flight status: [analyze]
- Weather conditions: [analyze]
- Crew status: [analyze]
- Passenger impact: [analyze]
- Predictions: [analyze]
- ATC status: [analyze]
- Alternate routes: [analyze]
- Cost analysis: [analyze]
- Compliance status: [analyze]

STEP 3: RISK EVALUATION
- List all identified risks with severity (High/Medium/Low)
- Explain the likelihood of each risk
- Identify which risks are most critical

STEP 4: OPTIONS ANALYSIS
- Option A: [describe action, cost, timeline, pros, cons]
- Option B: [describe action, cost, timeline, pros, cons]
- Option C: [describe action, cost, timeline, pros, cons]

STEP 5: TRADE-OFF ASSESSMENT
- Compare each option against the priority: {priority.upper()}
- Analyze pros and cons of each option
- Identify the optimal balance

STEP 6: DECISION RATIONALE
- Which option is recommended and WHY
- How it addresses the priority
- Why other options were rejected
- Confidence level in this decision

---
**FINAL RECOMMENDATION:**

Provide your final decision as a JSON object with these exact fields:

{{
    "primary_recommendation": "Clear, complete action statement describing exactly what to do",
    "secondary_actions": ["Action 1", "Action 2", "Action 3"],
    "risk_assessment": {{
        "level": "Low/Medium/High",
        "rationale": "Why this risk level"
    }},
    "estimated_cost": "Specific cost estimate with breakdown",
    "passenger_impact": {{
        "current_impact": "Current passenger situation",
        "expected_impact": "Expected after actions",
        "mitigation": "How to help passengers"
    }},
    "timeline": {{
        "immediate_actions": "0-30 minutes",
        "short_term": "30-120 minutes",
        "medium_term": "2-6 hours"
    }},
    "justification": "Detailed reasoning summary"
}}
"""

        print("\n" + "=" * 70)
        print("🧠 GLM-5.2 THINKING MODE: Generating structured reasoning...")
        print("=" * 70)
        print("⏳ Model is analyzing the problem step by step...\n")

        start_time = time.time()

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": """You are a senior aviation operations manager.
                        You ALWAYS show your complete step-by-step reasoning before providing any recommendation.
                        You are thorough, analytical, and structured in your thinking."""
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=3500,
                top_p=0.9
            )

            total_time = time.time() - start_time
            full_response = response.choices[0].message.content

            # Extract thinking
            thinking = self._extract_thinking(full_response)
            thinking_words = len(thinking.split())
            has_structured = any(h.lower() in thinking.lower() for h in self.section_headers)

            # Extract decision
            decision = self._extract_decision(full_response)

            print(f"⏱️ Response generated in {total_time:.2f} seconds")
            print(f"📝 Thinking depth: {thinking_words} words")
            print(f"📊 Structured reasoning: {'✅ Yes' if has_structured else '⚠️ Partial'}")

            # Display thinking summary
            print("\n" + "─" * 70)
            print("💭 THINKING PROCESS (Summary):")
            print("─" * 70)
            thinking_summary = thinking[:500] + ("..." if len(thinking) > 500 else "")
            print(thinking_summary)
            print("─" * 70)

            return ThinkingResult(
                thinking=thinking,
                thinking_words=thinking_words,
                has_structured=has_structured,
                total_time=total_time,
                priority=priority,
                timestamp=datetime.now().isoformat(),
                model_used=self.model,
                recommendation=decision.get("primary_recommendation", "See thinking"),
                risk_assessment=decision.get("risk_assessment", {"level": "Medium", "rationale": "See reasoning"}),
                estimated_cost=decision.get("estimated_cost", "$0"),
                justification=decision.get("justification", "See thinking"),
                timeline=decision.get("timeline", {"immediate": "See thinking", "short_term": "See thinking", "medium_term": "See thinking"})
            )

        except Exception as e:
            print(f"❌ GLM thinking failed: {e}")
            return ThinkingResult(
                thinking=f"Error: {str(e)}",
                thinking_words=0,
                has_structured=False,
                total_time=time.time() - start_time,
                priority=priority,
                timestamp=datetime.now().isoformat(),
                model_used="glm-5.2 (error)",
                recommendation="See error details",
                risk_assessment={"level": "High", "rationale": f"Error: {str(e)}"},
                estimated_cost="$0",
                justification=f"Error occurred: {str(e)}",
                timeline={"immediate": "Escalate", "short_term": "Review", "medium_term": "Resolve"}
            )

    def _extract_thinking(self, response: str) -> str:
        """Extract the thinking section from response"""
        patterns = [
            r'(?:STEP \d+:.*?)(?=FINAL RECOMMENDATION|---\s*$|\\Z)',
            r'(?:THINKING|REASONING)\s*[:：]\s*([\s\S]*?)(?:FINAL RECOMMENDATION|---\s*$)',
        ]

        for pattern in patterns:
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                thinking = match.group(1) if match.groups() else match.group(0)
                thinking = thinking.strip()
                if len(thinking) > 100:
                    return thinking

        return response[:1500]

    def _extract_decision(self, response: str) -> Dict:
        """Extract decision JSON from response"""
        # Try to find JSON
        json_patterns = [
            r'```json\s*({[\s\S]*?})\s*```',
            r'```\s*({[\s\S]*?})\s*```',
            r'({[\s\S]*"primary_recommendation"[\s\S]*})',
            r'({[\s\S]*"recommendation"[\s\S]*})',
        ]

        for pattern in json_patterns:
            match = re.search(pattern, response, re.DOTALL)
            if match:
                json_str = match.group(1) if match.groups() else match.group(0)
                try:
                    # Clean and parse
                    json_str = re.sub(r',\s*}', '}', json_str)
                    json_str = re.sub(r',\s*]', ']', json_str)
                    return json.loads(json_str)
                except:
                    continue

        return {
            "primary_recommendation": "See thinking output for recommendation",
            "secondary_actions": ["See detailed reasoning"],
            "risk_assessment": {"level": "Medium", "rationale": "See thinking"},
            "estimated_cost": "See reasoning",
            "passenger_impact": {"current": "See thinking", "expected": "See thinking", "mitigation": "See thinking"},
            "timeline": {"immediate": "See thinking", "short_term": "See thinking", "medium_term": "See thinking"},
            "justification": "Full justification in thinking above"
        }

# ============================================================================
# CELL 5: GPT-5.6 ENGINE
# ============================================================================

class GPT56Engine:
    """GPT-5.6 Engine with SOL, TERRA, LUNA variants"""

    def __init__(self, api_key: str):
        self.client = OpenAI(api_key=api_key)
        self.models = {
            "sol": "gpt-5.6-sol",
            "terra": "gpt-5.6-terra",
            "luna": "gpt-5.6-luna"
        }

        # Decision templates
        self.templates = {
            "flight_delay": """
You are an AOCC agent managing flight delays.

FLIGHT: {flight_number}
DEPARTURE: {departure} → {arrival}
SCHEDULED: {departure_time}
DELAY: {delay} minutes
PASSENGERS: {passengers}
CREW STATUS: {crew_status}
WEATHER: {weather}
ATC UPDATE: {atc_update}

Provide:
1. IMMEDIATE ACTIONS (0-30 min)
2. SHORT TERM ACTIONS (30-120 min)
3. PASSENGER COMMUNICATION PLAN
4. COST IMPACT ASSESSMENT
5. SUCCESS METRICS
""",
            "crew_unavailable": """
You are an AOCC agent managing crew issues.

FLIGHT: {flight_number}
CREW ROLE: {role}
ISSUE: {issue}
TIME TO DEPARTURE: {time_to_departure}
STANDBY CREW: {standby_available}

Provide:
1. CREW REPLACEMENT PLAN
2. REST COMPLIANCE CHECK
3. IMPACT ASSESSMENT
4. RECOVERY ACTIONS
""",
            "weather": """
You are an AOCC agent managing weather disruptions.

FLIGHT: {flight_number}
WEATHER: {condition}
AIRPORT: {airports}
FORECAST: {forecast}

Provide:
1. ROUTE ALTERNATIVES
2. TIMELINE ADJUSTMENTS
3. SAFETY ASSESSMENT
4. PASSENGER NOTIFICATION PLAN
""",
            "maintenance": """
You are an AOCC agent managing maintenance issues.

FLIGHT: {flight_number}
AIRCRAFT: {aircraft}
ISSUE: {issue}
ESTIMATED TIME: {estimated_time}

Provide:
1. MAINTENANCE PLAN
2. IMPACT ASSESSMENT
3. RECOVERY ACTIONS
4. COMMUNICATION PLAN
""",
            "regulatory": """
You are an AOCC agent managing regulatory compliance.

FLIGHT: {flight_number}
REGULATION: {regulation}
DEADLINE: {deadline}
CURRENT STATUS: {status}

Provide:
1. COMPLIANCE PLAN
2. TIMELINE FOR RESOLUTION
3. RISK ASSESSMENT
4. ESCALATION PLAN
"""
        }

        print("🤖 GPT-5.6 Engine initialized")
        print(f"   Models: {', '.join(self.models.values())}")

    def route_incident(self, incident: Incident) -> str:
        """Route incident to the best GPT-5.6 model"""

        # Calculate complexity
        complexity = {
            IncidentType.REGULATORY: 9,
            IncidentType.MAINTENANCE: 9,
            IncidentType.CREW_UNAVAILABLE: 8,
            IncidentType.FLIGHT_DELAY: 7,
            IncidentType.WEATHER: 6,
            IncidentType.ATC: 6,
            IncidentType.FUEL: 5,
            IncidentType.PASSENGER_ISSUE: 4,
            IncidentType.SECURITY: 8,
            IncidentType.MEDICAL: 7
        }.get(incident.type, 5)

        urgency = {
            "CRITICAL": 10,
            "HIGH": 8,
            "MEDIUM": 5,
            "LOW": 3
        }.get(incident.severity, 5)

        # Routing decision
        if incident.type in [IncidentType.REGULATORY, IncidentType.SECURITY]:
            return "sol"
        elif complexity >= 8 or urgency >= 9:
            return "sol"
        elif complexity >= 6 and urgency >= 6:
            return "terra"
        elif complexity >= 4:
            return "terra"
        else:
            return "luna"

    def get_context_data(self, incident: Incident) -> Dict:
        """Get context data for prompt template"""
        extra = incident.extra_data
        flight = incident.flight

        return {
            "flight_number": flight.flight_number,
            "departure": flight.departure,
            "arrival": flight.arrival,
            "departure_time": flight.departure_time,
            "passengers": flight.passengers,
            "aircraft": flight.aircraft,
            "delay": extra.get("delay", 0),
            "crew_status": extra.get("crew_status", "Available"),
            "weather": extra.get("weather", "Clear"),
            "atc_update": extra.get("atc", "Normal"),
            "role": extra.get("role", "Unknown"),
            "issue": incident.description,
            "time_to_departure": extra.get("time_to_departure", "2 hours"),
            "standby_available": extra.get("standby", "No"),
            "condition": extra.get("condition", "Unknown"),
            "airports": extra.get("airports", flight.arrival),
            "forecast": extra.get("forecast", "Improving"),
            "estimated_time": extra.get("estimated_time", "3 hours"),
            "regulation": extra.get("regulation", "FAA Part 121"),
            "deadline": extra.get("deadline", "24 hours"),
            "status": flight.status
        }

    def generate_decision(self, incident: Incident, thinking_result: Optional[ThinkingResult] = None) -> Decision:
        """Generate decision using GPT-5.6"""

        model_key = self.route_incident(incident)
        model = self.models[model_key]

        # Get template
        template = self.templates.get(
            incident.type.value,
            self.templates["flight_delay"]
        )

        # Build prompt
        context = self.get_context_data(incident)
        prompt = template.format(**context)

        # Add GLM thinking context if available
        if thinking_result:
            prompt = f"""
{thinking_result.thinking}

Based on the above thinking process, now provide a concise operational decision:

{prompt}
"""

        print(f"\n🤔 Generating decision with GPT-5.6-{model_key.upper()}...")
        start_time = time.time()

        try:
            response = self.client.responses.create(
                model=model,
                input=prompt
            )

            execution_time = time.time() - start_time
            response_text = response.output_text

            print(f"✅ Decision generated in {execution_time:.2f}s")

            return self._parse_decision(incident, response_text, model, execution_time, thinking_result)

        except Exception as e:
            print(f"❌ Error: {e}")
            return self._fallback_decision(incident, str(e))

    def _parse_decision(self, incident: Incident, response: str, model: str, execution_time: float, thinking_result: Optional[ThinkingResult]) -> Decision:
        """Parse GPT-5.6 response into Decision object"""

        lines = response.split('\n')
        actions = []
        recommendation = ""

        for line in lines:
            cleaned = line.strip()
            if cleaned and (cleaned.startswith(('•', '-', '*', '1.', '2.', '3.', '4.', '5.'))):
                actions.append(cleaned)
            elif "recommend" in cleaned.lower() or "suggest" in cleaned.lower():
                if recommendation == "":
                    recommendation = cleaned

        if not actions:
            sentences = response.split('.')
            actions = [s.strip() for s in sentences[:3] if len(s.strip()) > 10]

        if not recommendation:
            recommendation = actions[0] if actions else "See reasoning"

        priority_map = {
            "CRITICAL": Priority.CRITICAL,
            "HIGH": Priority.HIGH,
            "MEDIUM": Priority.MEDIUM,
            "LOW": Priority.LOW
        }
        priority = priority_map.get(incident.severity, Priority.MEDIUM)

        return Decision(
            incident_id=incident.id,
            recommendation=recommendation[:200],
            actions=actions[:10],
            priority=priority,
            estimated_cost=self._estimate_cost(incident),
            timeline=self._estimate_timeline(incident),
            risk_level=self._assess_risk(incident),
            confidence=0.85 if len(actions) > 3 else 0.70,
            reasoning=response[:500],
            model_used=model,
            timestamp=datetime.now().isoformat(),
            execution_time=execution_time,
            thinking_result=thinking_result
        )

    def _estimate_cost(self, incident: Incident) -> float:
        base = {
            "CRITICAL": 50000,
            "HIGH": 20000,
            "MEDIUM": 5000,
            "LOW": 1000
        }.get(incident.severity, 5000)
        return base + incident.flight.passengers * 150 + random.uniform(-5000, 5000)

    def _estimate_timeline(self, incident: Incident) -> str:
        return {
            "CRITICAL": "30-60 minutes",
            "HIGH": "60-120 minutes",
            "MEDIUM": "2-4 hours",
            "LOW": "4-24 hours"
        }.get(incident.severity, "2-4 hours")

    def _assess_risk(self, incident: Incident) -> str:
        return {
            "CRITICAL": "HIGH",
            "HIGH": "MEDIUM-HIGH",
            "MEDIUM": "MEDIUM",
            "LOW": "LOW"
        }.get(incident.severity, "MEDIUM")

    def _fallback_decision(self, incident: Incident, error: str) -> Decision:
        return Decision(
            incident_id=incident.id,
            recommendation="Manual intervention required",
            actions=["Escalate to supervisor", "Review manually", "Document error"],
            priority=Priority.HIGH,
            estimated_cost=10000,
            timeline="Immediate",
            risk_level="HIGH",
            confidence=0.3,
            reasoning=f"Fallback: {error[:200]}",
            model_used="fallback",
            timestamp=datetime.now().isoformat(),
            execution_time=0.0
        )

# ============================================================================
# CELL 6: AOC HYBRID AGENT - THE INTEGRATION
# ============================================================================

class AOCHybridAgent:
    """
    The main integration class that combines:
    1. GLM-5.2 for thinking/reasoning
    2. GPT-5.6 (SOL/TERRA/LUNA) for execution/decision
    3. Intelligent routing for optimal performance
    """

    def __init__(self):
        # Initialize both engines
        self.glm = GLMThinkingEngine(os.environ.get('ZAI_KEY', ''))
        self.gpt56 = GPT56Engine(os.environ.get('OPENAI_API_KEY', ''))

        self.state = AgentState()

        # Configuration
        self.config = {
            "use_thinking": True,  # Enable GLM-5.2 thinking for complex tasks
            "thinking_threshold": 6,  # Complexity threshold for thinking mode
            "max_iterations": 3,
            "rate_limit_delay": 0.5
        }

        print("\n" + "=" * 80)
        print("🏢 AOC HYBRID AGENT INITIALIZED")
        print("=" * 80)
        print(f"✅ GLM-5.2: {'Connected' if self.glm else 'Not connected'}")
        print(f"✅ GPT-5.6: {'Connected' if self.gpt56 else 'Not connected'}")
        print(f"📋 Models available: SOL, TERRA, LUNA, GLM-5.2")
        print("=" * 80)

    def analyze_incident(self, incident: Incident, priority: str = "balanced") -> Decision:
        """Full incident analysis using both GLM and GPT-5.6"""

        print(f"\n{'='*80}")
        print(f"🚨 ANALYZING INCIDENT: {incident.id}")
        print(f"   Type: {incident.type.value}")
        print(f"   Severity: {incident.severity}")
        print(f"   Flight: {incident.flight.flight_number}")
        print(f"{'='*80}")

        # Step 1: Determine if thinking mode is needed
        need_thinking = self._should_use_thinking(incident)

        thinking_result = None

        # Step 2: Run GLM-5.2 thinking if needed
        if need_thinking:
            print(f"\n🧠 GLM-5.2 THINKING: Enabled for this incident")
            thinking_result = self.glm.think(incident, priority)

            # Display thinking summary
            print(f"\n📊 GLM Thinking Summary:")
            print(f"   Words: {thinking_result.thinking_words}")
            print(f"   Structured: {thinking_result.has_structured}")
            print(f"   Time: {thinking_result.total_time:.2f}s")
            print(f"   Recommendation: {thinking_result.recommendation[:100]}...")
        else:
            print(f"\n⚡ GLM-5.2 THINKING: Skipped (simple incident)")

        # Step 3: Generate decision with GPT-5.6
        print(f"\n🤖 GPT-5.6: Generating decision...")
        decision = self.gpt56.generate_decision(incident, thinking_result)

        # Step 4: Store in state
        self.state.decisions.append(decision)
        self.state.incidents.append(incident)

        # Step 5: Update metrics
        self._update_metrics(decision, thinking_result)

        return decision

    def _should_use_thinking(self, incident: Incident) -> bool:
        """Determine if GLM-5.2 thinking mode should be used"""

        if not self.config["use_thinking"]:
            return False

        # Complex incident types that benefit from thinking
        complex_types = [
            IncidentType.REGULATORY,
            IncidentType.MAINTENANCE,
            IncidentType.CREW_UNAVAILABLE,
            IncidentType.SECURITY,
            IncidentType.MEDICAL
        ]

        if incident.type in complex_types:
            return True

        # High severity
        if incident.severity in ["CRITICAL", "HIGH"]:
            return True

        # High complexity
        complexity = {
            IncidentType.FLIGHT_DELAY: 7,
            IncidentType.CREW_UNAVAILABLE: 8,
            IncidentType.WEATHER: 6,
            IncidentType.MAINTENANCE: 9,
            IncidentType.PASSENGER_ISSUE: 4,
            IncidentType.REGULATORY: 9,
            IncidentType.ATC: 6,
            IncidentType.FUEL: 5,
            IncidentType.SECURITY: 8,
            IncidentType.MEDICAL: 7
        }.get(incident.type, 5)

        return complexity >= self.config["thinking_threshold"]

    def _update_metrics(self, decision: Decision, thinking_result: Optional[ThinkingResult]):
        """Update system metrics"""

        if "decisions_by_model" not in self.state.metrics:
            self.state.metrics["decisions_by_model"] = {}
        self.state.metrics["decisions_by_model"][decision.model_used] = \
            self.state.metrics["decisions_by_model"].get(decision.model_used, 0) + 1

        if "total_incidents" not in self.state.metrics:
            self.state.metrics["total_incidents"] = 0
        self.state.metrics["total_incidents"] += 1

        if thinking_result:
            if "thinking_time" not in self.state.metrics:
                self.state.metrics["thinking_time"] = 0
            self.state.metrics["thinking_time"] += thinking_result.total_time

        self.state.last_update = datetime.now().isoformat()

    def process_batch(self, incidents: List[Incident], priority: str = "balanced") -> List[Decision]:
        """Process multiple incidents in batch"""

        decisions = []
        total_start = time.time()

        print(f"\n{'='*80}")
        print(f"📦 BATCH PROCESSING: {len(incidents)} incidents")
        print(f"{'='*80}")

        for i, incident in enumerate(incidents, 1):
            print(f"\n📋 Incident {i}/{len(incidents)}")
            decision = self.analyze_incident(incident, priority)
            decisions.append(decision)

            # Rate limiting
            if i < len(incidents):
                time.sleep(self.config["rate_limit_delay"])

        total_time = time.time() - total_start
        print(f"\n✅ Batch complete: {len(decisions)} decisions in {total_time:.2f}s")

        return decisions

    def get_summary(self, decision: Decision) -> str:
        """Get human-readable decision summary"""

        summary = f"""
═══════════════════════════════════════════════════════
📋 DECISION SUMMARY - {decision.incident_id}
═══════════════════════════════════════════════════════

🎯 RECOMMENDATION:
{decision.recommendation}

📌 ACTIONS:
"""
        for i, action in enumerate(decision.actions[:5], 1):
            summary += f"   {i}. {action[:80]}\n"

        summary += f"""
⚠️ PRIORITY: {decision.priority.name}
💰 ESTIMATED COST: ${decision.estimated_cost:,.2f}
⏱️ TIMELINE: {decision.timeline}
🎲 RISK LEVEL: {decision.risk_level}
📊 CONFIDENCE: {decision.confidence*100:.0f}%
🤖 MODEL USED: {decision.model_used}
⚡ EXECUTION TIME: {decision.execution_time:.2f}s
"""

        if decision.thinking_result:
            summary += f"""
🧠 GLM-5.2 THINKING:
   Words: {decision.thinking_result.thinking_words}
   Structured: {decision.thinking_result.has_structured}
   Time: {decision.thinking_result.total_time:.2f}s
   Recommendation: {decision.thinking_result.recommendation[:100]}...
"""

        summary += "═══════════════════════════════════════════════════════\n"
        return summary

    def get_dashboard(self) -> Dict:
        """Get system dashboard"""

        total_decisions = len(self.state.decisions)
        if total_decisions == 0:
            return {"status": "No data", "total_incidents": 0}

        successful = sum(1 for d in self.state.decisions if d.confidence >= 0.7)
        success_rate = successful / total_decisions if total_decisions > 0 else 0

        return {
            "status": "HEALTHY" if success_rate >= 0.9 else "DEGRADED",
            "total_incidents": len(self.state.incidents),
            "total_decisions": total_decisions,
            "success_rate": success_rate,
            "model_usage": self.state.metrics.get("decisions_by_model", {}),
            "thinking_time": self.state.metrics.get("thinking_time", 0),
            "last_update": self.state.last_update
        }

# ============================================================================
# CELL 7: SIMULATOR (for testing)
# ============================================================================

class AOCCSimulator:
    """Generate test incidents"""

    @staticmethod
    def create_flight(num: str, dep: str, arr: str) -> Flight:
        now = datetime.now()
        return Flight(
            flight_number=num,
            departure=dep,
            arrival=arr,
            departure_time=(now + timedelta(hours=random.randint(1, 3))).isoformat(),
            arrival_time=(now + timedelta(hours=random.randint(4, 7))).isoformat(),
            status=random.choice(["SCHEDULED", "BOARDING", "DELAYED"]),
            gate=random.choice(["A1", "B3", "C5", "D2", "E8", "F10"]),
            aircraft=random.choice(["B737", "A320", "B787", "A350", "B777"]),
            passengers=random.randint(80, 280),
            crew=[f"P{random.randint(100,999)}", f"C{random.randint(100,999)}", f"F{random.randint(100,999)}"]
        )

    @staticmethod
    def create_incident(flight: Flight, incident_type: IncidentType, severity: str) -> Incident:
        data = {
            IncidentType.FLIGHT_DELAY: {
                "description": f"Flight {flight.flight_number} delayed due to operational issues",
                "impact": f"{flight.passengers} passengers affected",
                "extra": {"delay": random.randint(30, 150), "crew_status": "Available", "weather": "Clear", "atc": "Normal"}
            },
            IncidentType.CREW_UNAVAILABLE: {
                "description": f"Crew member unavailable for {flight.flight_number}",
                "impact": "Flight may be delayed or cancelled",
                "extra": {"role": random.choice(["Pilot", "Copilot", "Cabin Crew"]), "standby": random.choice(["Yes", "No"]), "time_to_departure": "2 hours"}
            },
            IncidentType.WEATHER: {
                "description": f"Severe weather conditions at {flight.arrival}",
                "impact": "Potential arrival delays",
                "extra": {"condition": random.choice(["Thunderstorms", "Snow", "Fog", "High Winds"]), "forecast": random.choice(["Improving", "Worsening", "Stable"]), "airports": flight.arrival}
            },
            IncidentType.MAINTENANCE: {
                "description": f"Mechanical issue detected on {flight.aircraft}",
                "impact": "Aircraft requires inspection",
                "extra": {"issue": random.choice(["Engine sensor", "Hydraulic leak", "Avionics", "Landing gear"]), "estimated_time": f"{random.randint(2, 6)} hours"}
            },
            IncidentType.PASSENGER_ISSUE: {
                "description": "Passenger medical emergency",
                "impact": "May require flight diversion",
                "extra": {"condition": "Medical", "severity": "High"}
            },
            IncidentType.REGULATORY: {
                "description": "Regulatory compliance issue detected",
                "impact": "Fine potential if not resolved",
                "extra": {"regulation": random.choice(["FAA Part 121", "EASA OPS", "ICAO Annex 6"]), "deadline": f"{random.randint(12, 48)} hours"}
            },
            IncidentType.ATC: {
                "description": "ATC congestion at departure airport",
                "impact": "Departure slot delay",
                "extra": {"delay": f"{random.randint(15, 60)} minutes", "reason": "Traffic volume"}
            },
            IncidentType.FUEL: {
                "description": "Fuel price surge affecting profitability",
                "impact": "Cost increase for route",
                "extra": {"price_increase": f"{random.randint(5, 25)}%", "impact": "High"}
            },
            IncidentType.SECURITY: {
                "description": "Security concern detected",
                "impact": "Enhanced screening required",
                "extra": {"type": random.choice(["Suspicious package", "Unauthorized access", "Threat"]), "status": "Active"}
            },
            IncidentType.MEDICAL: {
                "description": "Medical emergency on board",
                "impact": "Potential diversion",
                "extra": {"condition": random.choice(["Cardiac", "Stroke", "Injury", "Allergic reaction"]), "severity": "High"}
            }
        }

        d = data.get(incident_type, data[IncidentType.FLIGHT_DELAY])

        return Incident(
            id=f"INC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000,9999)}",
            type=incident_type,
            severity=severity,
            flight=flight,
            description=d["description"],
            impact=d["impact"],
            timestamp=datetime.now().isoformat(),
            status="OPEN",
            owner=f"Agent-{random.randint(1, 5)}",
            extra_data=d.get("extra", {})
        )

# ============================================================================
# CELL 8: MAIN EXECUTION
# ============================================================================

def main():
    """Run the complete hybrid AOC system"""

    print("\n" + "=" * 80)
    print("🚀 AOC HYBRID SYSTEM: GPT-5.6 + GLM-5.2")
    print("=" * 80)

    # Initialize agent
    agent = AOCHybridAgent()
    simulator = AOCCSimulator()

    # Create test flights
    print("\n📋 Creating test flights...")
    flights = [
        simulator.create_flight("AA123", "JFK", "LAX"),
        simulator.create_flight("UA456", "ORD", "SFO"),
        simulator.create_flight("DL789", "ATL", "MIA"),
        simulator.create_flight("SW888", "LAS", "DEN"),
        simulator.create_flight("EK202", "DXB", "LHR"),
        simulator.create_flight("BA456", "LHR", "JFK"),
        simulator.create_flight("AF789", "CDG", "ORD"),
    ]

    for f in flights:
        agent.state.active_flights.append(f)
        print(f"  ✈️ {f.flight_number}: {f.departure} → {f.arrival} ({f.passengers} pax)")

    # Create diverse incidents
    print("\n🚨 Creating diverse incidents...")
    incidents = [
        # Complex incidents (will use GLM thinking + SOL/TERRA)
        simulator.create_incident(flights[0], IncidentType.REGULATORY, "CRITICAL"),
        simulator.create_incident(flights[1], IncidentType.CREW_UNAVAILABLE, "HIGH"),
        simulator.create_incident(flights[2], IncidentType.MAINTENANCE, "HIGH"),
        simulator.create_incident(flights[3], IncidentType.SECURITY, "HIGH"),
        simulator.create_incident(flights[4], IncidentType.MEDICAL, "CRITICAL"),

        # Medium complexity (may or may not use thinking)
        simulator.create_incident(flights[5], IncidentType.FLIGHT_DELAY, "MEDIUM"),
        simulator.create_incident(flights[6], IncidentType.WEATHER, "MEDIUM"),

        # Simple incidents (LUNA, no thinking)
        simulator.create_incident(flights[0], IncidentType.FUEL, "LOW"),
        simulator.create_incident(flights[1], IncidentType.ATC, "LOW"),
    ]

    for inc in incidents:
        print(f"  🔴 {inc.id}: {inc.type.value} - {inc.severity}")

    # Process incidents with hybrid system
    print("\n" + "=" * 80)
    print("🔄 PROCESSING INCIDENTS WITH HYBRID SYSTEM")
    print("=" * 80)

    for incident in incidents[:7]:  # Process first 7 to save time
        print(f"\n{'='*80}")
        print(f"📋 Processing: {incident.id}")
        print(f"{'='*80}")

        decision = agent.analyze_incident(incident, priority="balanced")

        # Display summary
        print(agent.get_summary(decision))

        # Brief pause
        time.sleep(0.5)

    # Dashboard
    print("\n" + "=" * 80)
    print("📊 SYSTEM DASHBOARD")
    print("=" * 80)

    dashboard = agent.get_dashboard()
    print(f"Status: {dashboard['status']}")
    print(f"Total Incidents: {dashboard['total_incidents']}")
    print(f"Total Decisions: {dashboard['total_decisions']}")
    print(f"Success Rate: {dashboard['success_rate']*100:.1f}%")
    print(f"Thinking Time: {dashboard['thinking_time']:.2f}s")
    print("\nModel Usage:")
    for model, count in dashboard['model_usage'].items():
        print(f"  • {model}: {count} decisions")

    # Save results
    results = {
        "timestamp": datetime.now().isoformat(),
        "config": agent.config,
        "dashboard": dashboard,
        "decisions": [
            {
                "incident_id": d.incident_id,
                "recommendation": d.recommendation,
                "actions": d.actions[:5],
                "cost": d.estimated_cost,
                "confidence": d.confidence,
                "model": d.model_used,
                "execution_time": d.execution_time,
                "has_thinking": d.thinking_result is not None
            }
            for d in agent.state.decisions
        ]
    }

    with open("hybrid_aocc_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n💾 Results saved to: hybrid_aocc_results.json")
    print("=" * 80)
    print("✅ HYBRID SYSTEM EXECUTION COMPLETE")
    print("=" * 80)

# ============================================================================
# CELL 9: RUN THE SYSTEM
# ============================================================================

if __name__ == "__main__":
    main()

🚀 AOC HYBRID SYSTEM INITIALIZING
✅ OpenAI API Key: **********8UMA
✅ Z.ai API Key: **********60z3
✅ All imports loaded

🚀 AOC HYBRID SYSTEM: GPT-5.6 + GLM-5.2
🧠 GLM-5.2 Thinking Engine initialized
🤖 GPT-5.6 Engine initialized
   Models: gpt-5.6-sol, gpt-5.6-terra, gpt-5.6-luna

🏢 AOC HYBRID AGENT INITIALIZED
✅ GLM-5.2: Connected
✅ GPT-5.6: Connected
📋 Models available: SOL, TERRA, LUNA, GLM-5.2

📋 Creating test flights...
  ✈️ AA123: JFK → LAX (142 pax)
  ✈️ UA456: ORD → SFO (191 pax)
  ✈️ DL789: ATL → MIA (206 pax)
  ✈️ SW888: LAS → DEN (273 pax)
  ✈️ EK202: DXB → LHR (194 pax)
  ✈️ BA456: LHR → JFK (224 pax)
  ✈️ AF789: CDG → ORD (148 pax)

🚨 Creating diverse incidents...
  🔴 INC-20260710-6590: regulatory - CRITICAL
  🔴 INC-20260710-6810: crew_unavailable - HIGH
  🔴 INC-20260710-3207: maintenance - HIGH
  🔴 INC-20260710-7686: security - HIGH
  🔴 INC-20260710-6588: medical - CRITICAL
  🔴 INC-20260710-2363: flight_delay - MEDIUM
  🔴 INC-20260710-4651: weather - MEDIUM
  🔴 INC-20260710-6